# 02b DistilBERT robustness on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Git-Kapish/ReviewLens/blob/main/notebooks/02b_robustness_colab.ipynb)

Thin notebook: all logic is in `src/reviewlens/` and `scripts/run_robustness.py`. It perturbs the test reviews (typos, upper and lower case, emoji, truncation, negation) and scores the fine-tuned DistilBERT on them. The baseline half is already done locally.

**Before you run**
1. Runtime > Change runtime type > **T4 GPU**.
2. Have `robustness_inputs.zip` ready (about 250 MB; made locally in `data/`). It holds `models/distilbert/` and `data/processed/test.parquet`.
3. Run all cells. On a T4 this takes a few minutes.
4. The last cell downloads `robustness_distilbert.json`. Put it in your local `reports/` folder.

In [ ]:
!if [ ! -d "/content/ReviewLens" ]; then git clone https://github.com/Git-Kapish/ReviewLens.git /content/ReviewLens; else cd /content/ReviewLens && git pull; fi
%cd /content/ReviewLens
!pip install -q -e ".[transformer]"

In [ ]:
import torch

print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Inputs
Upload `robustness_inputs.zip` (cell below), or set `USE_DRIVE = True` and put the zip in the top level of your Google Drive; a Drive copy is more reliable than a 250 MB browser upload.

In [ ]:
import io
import zipfile
from pathlib import Path

USE_DRIVE = False

if not Path("models/distilbert/model.safetensors").exists():
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        blob = Path("/content/drive/MyDrive/robustness_inputs.zip").read_bytes()
    else:
        from google.colab import files

        print("Upload robustness_inputs.zip ...")
        blob = next(iter(files.upload().values()))
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extractall(".")
print(sorted(p.as_posix() for p in Path(".").glob("models/distilbert/*")), Path("data/processed/test.parquet").exists())

## Run
Writes `reports/robustness_distilbert.json`. The log shows the agreement of this run's clean predictions with the Phase 3 predictions (expected 1.0000 or very close).

In [ ]:
!PYTHONPATH=src python scripts/run_robustness.py --model distilbert

In [ ]:
import json

r = json.load(open("reports/robustness_distilbert.json"))
print("agreement with Phase 3 predictions:", r["clean_agreement_with_phase3_predictions"])
for name, p in r["perturbations"].items():
    print(f"{name:10s} n={p['n_applicable']:5d}  {p['clean_macro_f1']:.4f} -> {p['perturbed_macro_f1']:.4f}  delta {p['delta']['point']:+.4f}")

## Download

In [ ]:
from google.colab import files

files.download("reports/robustness_distilbert.json")